#### Importing required libraries 

In [6]:
import pandas as pd
from sklearn.metrics import *
from utils import Hetero_Data_Processor_Transfer_Learning
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from torch_geometric.nn import GATConv, to_hetero
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from torch.nn.functional import cross_entropy
from sklearn.preprocessing import RobustScaler
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")

#### Testing a single load 

In [3]:
train_dataset = 'charlie_hebdo'
test_dataset = 'ottawashooting'


time_cut =60*3*24
processor = Hetero_Data_Processor_Transfer_Learning(train_dataset, test_dataset, time_cut=time_cut,test_size=0.3)
data = processor.process()

rumour
1    139
0    119
Name: count, dtype: int64


In [4]:
data

HeteroData(
  id={
    x=[2859, 106],
    y=[2859],
    train_mask=[2859],
    val_mask=[2859],
    test_mask=[2859],
  },
  reply_user_id={ x=[26437, 104] },
  (id, retweet, reply_user_id)={ edge_index=[2, 26437] },
  (reply_user_id, rev_retweet, id)={ edge_index=[2, 26437] }
)

In [7]:
class GAT(torch.nn.Module):
    """
    Graph Attention Network (GAT) model with two GATConv layers and a final
    linear projection layer.

    This architecture applies graph attention mechanisms to learn contextual
    node embeddings based on graph connectivity. Dropout is applied after each
    GAT layer to help reduce overfitting.

    Parameters
    ----------
    dim_h : int
        Number of hidden attention heads/features in the first GAT layer.
    dim_i : int
        Number of intermediate output features of the second GAT layer.
    dim_out : int
        Output feature dimension, typically corresponding to the number of
        prediction classes or embedding size.

    Attributes
    ----------
    conv1 : GATConv
        First graph attention convolution layer with learned attention weights.
    conv2 : GATConv
        Second graph attention convolution layer that refines node embeddings.
    linear : nn.Linear
        Fully connected layer to project the learned embeddings to the output dimension.
    dropout : nn.Dropout
        Dropout layer applied after each convolution to reduce overfitting.

    Forward Inputs
    --------------
    x : torch.Tensor
        Node feature matrix of shape [num_nodes, num_features].
    edge_index : torch.LongTensor
        Graph edge index tensor of shape [2, num_edges] defining connectivity.

    Returns
    -------
    torch.Tensor
        Output node representations of shape [num_nodes, dim_out].
    """

    def __init__(self, dim_h,dim_i, dim_out):
        super().__init__()
        self.conv1 = GATConv((-1, -1), dim_h, add_self_loops=False)
        self.conv2 = GATConv(dim_h, dim_i, add_self_loops=False)
        self.linear = nn.Linear(dim_i, dim_out)
        self.dropout = nn.Dropout(p=0.4)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index).relu()
        h = self.dropout(h)
        h = self.conv2(h, edge_index).relu()
        h = self.dropout(h)
        h = self.linear(h)
        return h


In [19]:
def evaluate(model, data, mask_names):

    """
    Evaluate a graph classification model using masked subsets of the data.

    This function runs the model in evaluation mode, computes predictions,
    filters them using one or multiple masks from the input dataset, and
    returns common binary classification metrics.

    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model producing class logits from graph inputs.
    data : torch_geometric.data.HeteroData
        Heterogeneous graph data structure containing:
        - `x_dict`: dictionary of node feature matrices
        - `edge_index_dict`: dictionary of edge connectivity
        - `'id'` node type with attributes `y` and boolean masks
          (e.g., 'train_mask', 'val_mask', 'test_mask')
    mask_names : str or list[str]
        Name(s) of mask attributes to evaluate on. If multiple masks
        are provided, they are combined using logical OR.

    Returns
    -------
    tuple(float, float, float, float)
        A tuple containing:
        - acc : float
            Accuracy score.
        - precision : float
            Proportion of predicted positives that are correctly classified.
        - recall : float
            True positive rate.
        - auc : float
            ROC-AUC score based on predicted class probabilities.

    Notes
    -----
    - Metrics are computed only on masked nodes.
    - If ROC-AUC cannot be computed due to a single class present in labels,
      a value of 0.0 is returned.
    """
    
    model.eval()
    out = model(data.x_dict, data.edge_index_dict)
    preds = out['id'].argmax(dim=1)
    labels = data['id'].y

    if isinstance(mask_names, str):
        mask = data['id'][mask_names]
    else:
        mask = torch.zeros_like(data['id'].y, dtype=torch.bool)
        for name in mask_names:
            mask |= data['id'][name]

    preds_masked = preds[mask]
    labels_masked = labels[mask]
    probs = out['id'][mask][:, 1]  # Positive class probability

    acc = accuracy_score(labels_masked.cpu(), preds_masked.cpu())
    precision = precision_score(labels_masked.cpu(), preds_masked.cpu(), zero_division=0)
    recall = recall_score(labels_masked.cpu(), preds_masked.cpu(), zero_division=0)

    try:
        auc = roc_auc_score(labels_masked.cpu(), probs.detach().cpu())
    except ValueError:
        auc = 0.0

    return acc, precision, recall, auc



def train(model, data, optimizer, epochs=100):
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        out = model(data.x_dict, data.edge_index_dict)
        out_id = out['id']
        loss = cross_entropy(out_id[data['id'].train_mask], data['id'].y[data['id'].train_mask])
        loss.backward()
        optimizer.step()

        # Train metrics
        acc, precision, recall, auc = evaluate(model, data, 'train_mask')
        print(f"[Epoch {epoch:03d}] Train - Acc: {acc:.4f} | Prec: {precision:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")

        # Val metrics every 10 epochs
        if epoch % 10 == 0:
            acc_val, prec_val, recall_val, auc_val = evaluate(model, data, 'val_mask')
            print(f"[Epoch {epoch:03d}] Val   - Acc: {acc_val:.4f} | Prec: {prec_val:.4f} | Recall: {recall_val:.4f} | AUC: {auc_val:.4f}")

    print("\nFinal Evaluation (Val + Test):")
    acc_final, prec_final, recall_final, auc_final = evaluate(model, data, ['val_mask', 'test_mask'])
    print(f"[Final] Val+Test - Acc: {acc_final:.4f} | Prec: {prec_final:.4f} | Recall: {recall_final:.4f} | AUC: {auc_final:.4f}")
    print("\nFinal Evaluation (Test):")
    acc_final, prec_final, recall_final, auc_final = evaluate(model, data, ['test_mask'])
    print(f"[Final] Test - Acc: {acc_final:.4f} | Prec: {prec_final:.4f} | Recall: {recall_final:.4f} | AUC: {auc_final:.4f}")

#### Example  training

In [20]:
model = GAT(dim_h=64, dim_i=32, dim_out=2)
model = to_hetero(model, data.metadata(), aggr='sum')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data, model = data.to(device), model.to(device)

In [21]:
train(model, data, optimizer, epochs=100)


[Epoch 001] Train - Acc: 0.6717 | Prec: 0.1654 | Recall: 0.0288 | AUC: 0.5865
[Epoch 002] Train - Acc: 0.6790 | Prec: 0.1354 | Recall: 0.0170 | AUC: 0.6360
[Epoch 003] Train - Acc: 0.6886 | Prec: 0.1429 | Recall: 0.0118 | AUC: 0.6816
[Epoch 004] Train - Acc: 0.6920 | Prec: 0.0714 | Recall: 0.0039 | AUC: 0.7034
[Epoch 005] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.7192
[Epoch 006] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.7316
[Epoch 007] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.7412
[Epoch 008] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.7500
[Epoch 009] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.7594
[Epoch 010] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.7684
[Epoch 010] Val   - Acc: 0.5116 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6188
[Epoch 011] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.7757
[Epoch 012] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 

#### Setting MLflow Experiment

In [22]:
mlflow.set_experiment("GAT Network 2025-11-04 Ottawa Shooting TF")

2025/11/09 17:54:37 INFO mlflow.tracking.fluent: Experiment with name 'GAT Network 2025-11-04 Ottawa Shooting TF' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/rumour-detection-gnn/New experiments/mlruns/93', creation_time=1762710877244, experiment_id='93', last_update_time=1762710877244, lifecycle_stage='active', name='GAT Network 2025-11-04 Ottawa Shooting TF', tags={}>

#### Loading dataset statistics to get the final time cut 

In [25]:
df_posts_by_time_cut = pd.read_csv('ottawa_shooting_posts_by_time_cut.csv')

In [26]:
time_cut_last_post = int(df_posts_by_time_cut[df_posts_by_time_cut.post==\
                         int(df_posts_by_time_cut['post'].max())].time_cut.min())

In [27]:
time_cut_last_post

599

**Creating evaluate_metrics function to assess classification when new posts are created**

In [28]:
def evaluate_metrics(model, data, mask):


    """
    Compute evaluation metrics for a model on a masked node subset.

    The function performs prediction using the trained model, extracts
    predictions over a specific boolean mask, and computes standard
    classification metrics including accuracy, macro precision, macro
    recall, and ROC-AUC.

    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model that outputs logits for the `'id'` node type.
    data : torch_geometric.data.HeteroData
        Heterogeneous graph data containing node features, edge connections,
        labels, and a boolean mask to filter evaluation nodes.
    mask : torch.Tensor or list[bool]
        Boolean mask selecting the subset of nodes to evaluate.

    Returns
    -------
    tuple(float, float, float, float)
        A tuple of:
        - acc : float
            Accuracy score.
        - prec : float
            Macro-averaged precision.
        - recall : float
            Macro-averaged recall.
        - auc : float
            ROC-AUC score based on probability of the positive class.

    Notes
    -----
    - Evaluation is performed inside a `torch.no_grad()` block to disable gradient tracking.
    - If ROC-AUC computation fails (e.g., only one class present), a value of 0.0 is returned.
    """


    model.eval()
    with torch.no_grad():
        out = model(data.x_dict, data.edge_index_dict)['id']
        preds = out.argmax(dim=1)
        probs = out[:, 1]  # Probability of class 1

    true = data['id'].y[mask]
    pred = preds[mask]
    prob = probs[mask]

    acc = accuracy_score(true.cpu(), pred.cpu())
    prec = precision_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)
    recall = recall_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)
    try:
        auc = roc_auc_score(true.cpu(), prob.cpu())
    except:
        auc = 0.0

    return acc, prec, recall, auc

* **The initial  time cut will be 10 minutes after the first post publication**
*  **The final time cut will be equal to 6 hours after the publication of last post**

In [29]:



previous_node_count = 0  # Start with no nodes

for time_cut in range(40, time_cut_last_post+(60*24*1), 30):
    print(f"\n=== Time Cut: {time_cut} ===")
    train_dataset = 'charlie_hebdo'
    #test_dataset = 'ferguson'
    #test_dataset = 'sydneysiege'
    test_dataset = 'ottawashooting'
    #test_dataset = 'germanwings_crash'
    time_cut =time_cut
    processor = Hetero_Data_Processor_Transfer_Learning(train_dataset, test_dataset, time_cut=time_cut,test_size=0.3)
    data = processor.process()

    current_node_count = data['id'].x.shape[0]
    new_node_indices = np.arange(previous_node_count, current_node_count)
    previous_node_count = current_node_count

    model = GAT(dim_h=64, dim_i=32, dim_out=2)
    model = to_hetero(model, data.metadata(), aggr='sum')
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data, model = data.to(device), model.to(device)

    # Compute imbalance
    y_train = data['id'].y[data['id'].train_mask].cpu()
    imbalance = (y_train == 1).sum() / len(y_train)

    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        for epoch in range(1, 101):
            model.train()
            optimizer.zero_grad()
            out = model(data.x_dict, data.edge_index_dict)['id']
            mask = data['id'].train_mask
            loss = F.cross_entropy(out[mask], data['id'].y[mask])
            loss.backward()
            optimizer.step()

            if epoch % 100 == 0:
                 train_acc, train_prec, train_recall, train_auc= evaluate_metrics(model, data, data['id'].train_mask)
                 print(f"[Epoch {epoch}] Train Loss: {loss:.4f} | Train Recall: {train_recall:.4f} | Train Auc: {train_auc:.4f}")
                
        # Evaluate all predictions
        model.eval()
        with torch.no_grad():
            out = model(data.x_dict, data.edge_index_dict)['id']
            preds = out.argmax(dim=1)
            probs = out[:, 1]
    
        # New instances in val/test set
        val_test_mask = (data['id'].val_mask | data['id'].test_mask).cpu().numpy()
        new_instance_mask = np.zeros_like(val_test_mask, dtype=bool)
        new_instance_mask[new_node_indices] = True
        final_mask = new_instance_mask & val_test_mask
    
        if final_mask.sum() > 0:
            # Compute metrics
            true_new = data['id'].y.cpu().numpy()[final_mask]
            pred_new = preds.cpu().numpy()[final_mask]
            prob_new = probs.cpu().numpy()[final_mask]
        
            new_precision = precision_score(true_new, pred_new, average='macro', zero_division=0)
            new_recall = recall_score(true_new, pred_new, average='macro', zero_division=0)
            new_acc = accuracy_score(true_new, pred_new)
        else:
            new_precision = 0
            new_recall = 0
            new_acc =0
            print("No new instances to evaluate.")
            
    
        # Compute metrics
        
        all_eval_mask = data['id'].val_mask | data['id'].test_mask
        acc, prec, recall, auc = evaluate_metrics(model, data, all_eval_mask)
        
        print(f"[Final Val+Test] Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")
    
    
        print(f"New Instances: {final_mask.sum()}")
        print(f"New Precision: {new_precision:.4f} | New Recall: {new_recall:.4f}")

        mlflow.log_metric("new_posts", final_mask.sum())
        
        mlflow.log_metric("final_precision", prec)
        mlflow.log_metric("final_recall", recall)
        mlflow.log_metric("final_auc", auc)
        mlflow.log_metric("final_acc", acc)

        mlflow.log_metric("curr_precision", new_precision)
        
        mlflow.log_metric("curr_recall", new_recall)
        mlflow.log_metric("curr_acc", new_acc)

        mlflow.log_metric("time_cut", time_cut)


=== Time Cut: 40 ===
rumour
0    24
1     8
Name: count, dtype: int64
[Epoch 100] Train Loss: 0.3751 | Train Recall: 0.8066 | Train Auc: 0.9045
[Final Val+Test] Acc: 0.8750 | Prec: 0.8273 | Recall: 0.8750 | AUC: 0.9479
New Instances: 32
New Precision: 0.8273 | New Recall: 0.8750

=== Time Cut: 70 ===
rumour
0    29
1    18
Name: count, dtype: int64


KeyboardInterrupt: 